# LeetCode 3559 — Number of Ways to Assign Edge Weights II bằng Python

- Link: https://leetcode.com/problems/number-of-ways-to-assign-edge-weights-ii/
- Độ khó: **Hard**
- Chủ đề: **Tổ hợp / tính chẵn lẻ** (đếm cách gán 1 hoặc 2 để tổng lẻ), **khoảng cách trên cây bằng LCA** (binary lifting, Tarjan offline), **lũy thừa modulo**

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def assignEdgeWeights(self, edges: List[List[int]], queries: List[List[int]]) -> List[int]:
```

## 1. Mục tiêu học

1. Đếm số cách gán trọng số `1` hoặc `2` cho `L` cạnh để **tổng lẻ** bằng DP `(độ dài, chẵn lẻ)` rồi rút ra công thức đóng `2^(L−1)` (với `L ≥ 1`).
2. Hiểu vì sao: trọng số 2 không đổi chẵn lẻ, trọng số 1 đảo chẵn lẻ ⇒ số cách tổng lẻ = số tập con có **lẻ** phần tử.
3. Tính **số cạnh trên đường đi** `u → v` bằng `dep[u] + dep[v] − 2·dep[lca]`.
4. Cài LCA bằng **binary lifting** và bằng **Tarjan offline** (DSU), so sánh hai cách.
5. Dùng `pow(2, L − 1, MOD)` hoặc bảng lũy thừa tính sẵn.

## 2. Đề bài & ví dụ

Cây vô hướng `n` đỉnh (nhãn `1..n`), gốc 1. Mọi cạnh ban đầu có trọng số 0; ta phải gán **mỗi cạnh** trọng số `1` hoặc `2`.

Với mỗi truy vấn `[u, v]`: chỉ xét các cạnh trên **đường đi** `u → v`; đếm số cách gán trọng số cho các cạnh đó sao cho **tổng trọng số lẻ**. Kết quả modulo `10^9 + 7`.

**Ví dụ 1:** `edges=[[1,2]]`, `queries=[[1,1],[1,2]]` → `[0,1]`.

**Ví dụ 2:** `edges=[[1,2],[1,3],[3,4],[3,5]]`, `queries=[[1,4],[3,4],[2,5]]` → `[2,1,4]`.

**Ràng buộc:** `2 ≤ n ≤ 10^5`, `1 ≤ len(queries) ≤ 10^5`.

In [1]:
from typing import List
MOD = 10**9 + 7
examples = [
    ([[1, 2]], [[1, 1], [1, 2]], [0, 1]),
    ([[1, 2], [1, 3], [3, 4], [3, 5]], [[1, 4], [3, 4], [2, 5]], [2, 1, 4]),
]
for e, q, exp in examples:
    print(e, q, '-> kỳ vọng', exp)

[[1, 2]] [[1, 1], [1, 2]] -> kỳ vọng [0, 1]
[[1, 2], [1, 3], [3, 4], [3, 5]] [[1, 4], [3, 4], [2, 5]] -> kỳ vọng [2, 1, 4]


## 3. Kiến thức nền

### 3.1. Chẵn lẻ của tổng

`1` là số lẻ, `2` là số chẵn. Tổng của các trọng số lẻ ⇔ **số cạnh mang trọng số 1 là số lẻ**.

In [2]:
from itertools import product
L = 3
for ws in product([1, 2], repeat=L):
    print(ws, 'tổng =', sum(ws), 'lẻ' if sum(ws) % 2 else 'chẵn', '| số cạnh trọng số 1 =', ws.count(1))

(1, 1, 1) tổng = 3 lẻ | số cạnh trọng số 1 = 3
(1, 1, 2) tổng = 4 chẵn | số cạnh trọng số 1 = 2
(1, 2, 1) tổng = 4 chẵn | số cạnh trọng số 1 = 2
(1, 2, 2) tổng = 5 lẻ | số cạnh trọng số 1 = 1
(2, 1, 1) tổng = 4 chẵn | số cạnh trọng số 1 = 2
(2, 1, 2) tổng = 5 lẻ | số cạnh trọng số 1 = 1
(2, 2, 1) tổng = 5 lẻ | số cạnh trọng số 1 = 1
(2, 2, 2) tổng = 6 chẵn | số cạnh trọng số 1 = 0


### 3.2. Số tập con có lẻ phần tử

Với tập `L ≥ 1` phần tử, số tập con lẻ phần tử = số tập con chẵn phần tử = `2^(L−1)`. Lý do: ghép cặp mỗi tập con với tập con "thêm/bớt phần tử đầu tiên" — mỗi cặp có đúng một tập chẵn và một tập lẻ. Với `L = 0` chỉ có tập rỗng (chẵn) ⇒ 0 tập lẻ.

In [3]:
from math import comb
for L in range(0, 7):
    odd = sum(comb(L, k) for k in range(1, L + 1, 2))
    print(f"L={L}: tập con lẻ = {odd}, 2^(L-1) = {2 ** (L - 1) if L else 0}")

L=0: tập con lẻ = 0, 2^(L-1) = 0
L=1: tập con lẻ = 1, 2^(L-1) = 1
L=2: tập con lẻ = 2, 2^(L-1) = 2
L=3: tập con lẻ = 4, 2^(L-1) = 4
L=4: tập con lẻ = 8, 2^(L-1) = 8
L=5: tập con lẻ = 16, 2^(L-1) = 16
L=6: tập con lẻ = 32, 2^(L-1) = 32


### 3.3. Lũy thừa modulo

`pow(2, e, MOD)` tính `2^e mod MOD` trong `O(log e)`. Khi có nhiều truy vấn, dựng sẵn bảng `p2[i] = 2^i mod MOD` cho `i ≤ n` là `O(1)` mỗi truy vấn.

In [4]:
p2 = [1] * 11
for i in range(1, 11):
    p2[i] = p2[i - 1] * 2 % MOD
print(p2, pow(2, 10**5, MOD))

[1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024] 607723520


### 3.4. Độ sâu, binary lifting, LCA

BFS từ gốc cho `par` và `dep`. `up[j][x]` là tổ tiên thứ `2^j`. LCA: đưa hai đỉnh về cùng độ sâu rồi nhảy song song. Số cạnh trên đường `u → v` là `dep[u] + dep[v] − 2·dep[lca]`.

In [5]:
def bfs(n, edges, root=0):
    adj = [[] for _ in range(n)]
    for a, b in edges:
        adj[a - 1].append(b - 1)
        adj[b - 1].append(a - 1)
    par, dep = [root] * n, [0] * n
    seen = [False] * n
    seen[root] = True
    order = [root]
    for u in order:
        for v in adj[u]:
            if not seen[v]:
                seen[v] = True
                par[v], dep[v] = u, dep[u] + 1
                order.append(v)
    return par, dep, order, adj

def build_up(par):
    n = len(par)
    LOG = max(1, (n - 1).bit_length())
    up = [par[:]]
    for _ in range(1, LOG):
        pr = up[-1]
        up.append([pr[pr[x]] for x in range(n)])
    return up

def lca(up, dep, a, b):
    if dep[a] < dep[b]:
        a, b = b, a
    diff, j = dep[a] - dep[b], 0
    while diff:
        if diff & 1:
            a = up[j][a]
        diff >>= 1
        j += 1
    if a == b:
        return a
    for j in range(len(up) - 1, -1, -1):
        if up[j][a] != up[j][b]:
            a, b = up[j][a], up[j][b]
    return up[0][a]

par, dep, order, adj = bfs(5, [[1, 2], [1, 3], [3, 4], [3, 5]])
up = build_up(par)
print('dep =', dep, '| lca(2,5) =', lca(up, dep, 1, 4) + 1)

dep = [0, 1, 1, 2, 2] | lca(2,5) = 1


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Mỗi truy vấn chỉ phụ thuộc **số cạnh `L`** trên đường đi (các cạnh khác bị bỏ qua, và cạnh nào cũng như nhau).

Tính tay ví dụ 2, truy vấn `[2, 5]`: đường `2 → 1 → 3 → 5`, `L = 3`. Các cách tổng lẻ: số cạnh mang trọng số 1 là 1 hoặc 3: `C(3,1) + C(3,3) = 3 + 1 = 4 = 2^2`. Khớp đề.

| truy vấn | đường | L | đáp án |
|---|---|---|---|
| [1,4] | 1→3→4 | 2 | 2 |
| [3,4] | 3→4 | 1 | 1 |
| [2,5] | 2→1→3→5 | 3 | 4 |

In [6]:
for u, v in [[1, 4], [3, 4], [2, 5]]:
    l = lca(up, dep, u - 1, v - 1)
    L = dep[u - 1] + dep[v - 1] - 2 * dep[l]
    print(f"[{u},{v}]: lca = {l + 1}, L = {L}, đáp án = {pow(2, L - 1, MOD) if L else 0}")

[1,4]: lca = 1, L = 2, đáp án = 2
[3,4]: lca = 3, L = 1, đáp án = 1
[2,5]: lca = 1, L = 3, đáp án = 4


## 5. Cách 1 — Brute Force: dựng đường đi, thử mọi cách gán

BFS từ `u` để tìm đường tới `v`, đếm `L`, rồi thử mọi `2^L` cách gán (chỉ dùng được khi đường ngắn).

In [7]:
from collections import deque

def path_len_bfs(adj, u, v):
    dist = {u: 0}
    dq = deque([u])
    while dq:
        x = dq.popleft()
        if x == v:
            return dist[x]
        for y in adj[x]:
            if y not in dist:
                dist[y] = dist[x] + 1
                dq.append(y)

def assign_brute(edges, queries):
    n = len(edges) + 1
    _, _, _, adj = bfs(n, edges)
    res = []
    for u, v in queries:
        L = path_len_bfs(adj, u - 1, v - 1)
        cnt = sum(1 for ws in product([1, 2], repeat=L) if sum(ws) % 2 == 1)
        res.append(cnt % MOD)
    return res

for e, q, exp in examples:
    print(assign_brute(e, q), '| kỳ vọng', exp)

[0, 1] | kỳ vọng [0, 1]
[2, 1, 4] | kỳ vọng [2, 1, 4]


**Độ phức tạp:** `O(n + 2^L · L)` mỗi truy vấn — bùng nổ khi `L` lớn (tới `10^5`).

**Nút thắt:** (1) thử `2^L` cách; (2) BFS lại cả cây cho mỗi truy vấn.

## 6. Ý tưởng thuật toán / bước đột phá

**Đột phá 1 — đếm không cần liệt kê:** DP `ways[L][p]` = số cách gán `L` cạnh để tổng có chẵn lẻ `p`:

`ways[L][p] = ways[L−1][p] (cạnh mới = 2) + ways[L−1][1−p] (cạnh mới = 1)`, `ways[0] = [1, 0]`.

Từ đó `ways[L][1] = 2^(L−1)` với `L ≥ 1` (mục 3.2).

**Đột phá 2 — số cạnh bằng LCA:** `L = dep[u] + dep[v] − 2·dep[lca(u, v)]`, LCA bằng binary lifting `O(log n)` mỗi truy vấn.

Quy tắc:

- **Nếu** `u = v` (L = 0) **thì** đáp án 0.
- **Ngược lại** đáp án `2^(L−1) mod (10^9 + 7)`.

In [8]:
def ways_table(Lmax):
    w = [[1, 0]]
    for L in range(1, Lmax + 1):
        e, o = w[-1]
        w.append([(e + o) % MOD, (o + e) % MOD])     # cạnh 2 giữ chẵn lẻ, cạnh 1 đảo
    return w

w = ways_table(6)
print([row[1] for row in w])   # 0, 1, 2, 4, 8, 16, 32

[0, 1, 2, 4, 8, 16, 32]


## 7. Các cách tiếp theo

### 7.1. Cách 2 — DP chẵn lẻ + leo cha để đo `L`

Bỏ việc liệt kê: dùng bảng `ways`. Đo `L` bằng cách leo cha (đưa về cùng độ sâu rồi leo song song) — `O(độ sâu)` mỗi truy vấn.

In [9]:
def assign_climb(edges, queries):
    n = len(edges) + 1
    par, dep, _, _ = bfs(n, edges)
    w = ways_table(n)
    res = []
    for u, v in queries:
        a, b = u - 1, v - 1
        L = 0
        while dep[a] > dep[b]:
            a = par[a]; L += 1
        while dep[b] > dep[a]:
            b = par[b]; L += 1
        while a != b:
            a, b = par[a], par[b]; L += 2
        res.append(w[L][1])
    return res

for e, q, exp in examples:
    print(assign_climb(e, q), '| kỳ vọng', exp)

[0, 1] | kỳ vọng [0, 1]
[2, 1, 4] | kỳ vọng [2, 1, 4]


**Độ phức tạp:** `O(n + q · h)` — xấu nhất `O(n q)` trên cây đường thẳng.

### 7.2. Cách 3 — Binary lifting + công thức `2^(L−1)` (lời giải chính)

In [10]:
def assign_lifting(edges, queries):
    n = len(edges) + 1
    par, dep, _, _ = bfs(n, edges)
    up = build_up(par)
    p2 = [1] * (n + 1)
    for i in range(1, n + 1):
        p2[i] = p2[i - 1] * 2 % MOD
    res = []
    for u, v in queries:
        l = lca(up, dep, u - 1, v - 1)
        L = dep[u - 1] + dep[v - 1] - 2 * dep[l]
        res.append(p2[L - 1] if L else 0)
    return res

for e, q, exp in examples:
    print(assign_lifting(e, q), '| kỳ vọng', exp)

[0, 1] | kỳ vọng [0, 1]
[2, 1, 4] | kỳ vọng [2, 1, 4]


**Độ phức tạp:** tiền xử lý `O(n log n)`, mỗi truy vấn `O(log n)`.

### 7.3. Cách 4 — LCA offline bằng thuật toán Tarjan (DSU)

Đọc hết truy vấn trước. DFS cây; khi **hoàn tất** đỉnh `u`, với mỗi truy vấn `(u, v)` mà `v` đã hoàn tất, LCA là `anc[find(v)]`. Sau khi xong `u`, gộp `u` vào cha và đặt tổ tiên đại diện là cha. Tổng `O((n + q) α(n))`.

In [11]:
def assign_tarjan(edges, queries):
    n = len(edges) + 1
    par, dep, order, adj = bfs(n, edges)
    qs = [[] for _ in range(n)]
    for i, (u, v) in enumerate(queries):
        qs[u - 1].append((v - 1, i))
        qs[v - 1].append((u - 1, i))
    dsu = list(range(n))

    def find(x):
        while dsu[x] != x:
            dsu[x] = dsu[dsu[x]]
            x = dsu[x]
        return x

    anc = list(range(n))
    done = [False] * n
    lca_of = [0] * len(queries)
    # thứ tự hậu tự: đi ngược BFS cũng cho "con xong trước cha", nhưng Tarjan cần DFS thật
    children = [[] for _ in range(n)]
    for v in order[1:]:
        children[par[v]].append(v)
    stack = [(0, 0)]
    while stack:
        u, i = stack.pop()
        if i < len(children[u]):
            stack.append((u, i + 1))
            stack.append((children[u][i], 0))
            continue
        # hoàn tất u
        done[u] = True
        for v, qi in qs[u]:
            if done[v]:
                lca_of[qi] = anc[find(v)]
        if u != 0:
            p = par[u]
            dsu[find(u)] = find(p)
            anc[find(p)] = p
    p2 = [1] * (n + 1)
    for i in range(1, n + 1):
        p2[i] = p2[i - 1] * 2 % MOD
    res = []
    for (u, v), l in zip(queries, lca_of):
        L = dep[u - 1] + dep[v - 1] - 2 * dep[l]
        res.append(p2[L - 1] if L else 0)
    return res

for e, q, exp in examples:
    print(assign_tarjan(e, q), '| kỳ vọng', exp)

[0, 1] | kỳ vọng [0, 1]
[2, 1, 4] | kỳ vọng [2, 1, 4]


Lưu ý truy vấn `u = v`: khi hoàn tất `u`, `done[u]` đã đúng nên `lca = anc[find(u)] = u` ⇒ `L = 0` — đúng.

**Độ phức tạp:** `O((n + q) α(n))`, không cần bảng `n log n`.

## 8. TRACE từng bước

Trace cách 3 trên ví dụ 2: in độ sâu, quá trình LCA (nâng đỉnh sâu, nhảy song song) và đáp án mỗi truy vấn.

In [12]:
def trace(edges, queries):
    n = len(edges) + 1
    par, dep, _, _ = bfs(n, edges)
    up = build_up(par)
    print('độ sâu (nhãn 1..n):', {i + 1: dep[i] for i in range(n)})
    for u, v in queries:
        a, b = u - 1, v - 1
        steps = []
        if dep[a] < dep[b]:
            a, b = b, a
        diff, j = dep[a] - dep[b], 0
        while diff:
            if diff & 1:
                steps.append(f"nâng {a + 1} lên 2^{j} -> {up[j][a] + 1}")
                a = up[j][a]
            diff >>= 1; j += 1
        if a != b:
            for j in range(len(up) - 1, -1, -1):
                if up[j][a] != up[j][b]:
                    steps.append(f"nhảy song song 2^{j}: {a + 1},{b + 1} -> {up[j][a] + 1},{up[j][b] + 1}")
                    a, b = up[j][a], up[j][b]
            a = up[0][a]
        L = dep[u - 1] + dep[v - 1] - 2 * dep[a]
        ans = pow(2, L - 1, MOD) if L else 0
        print(f"[{u},{v}]: {'; '.join(steps) or 'không cần nhảy'} | lca = {a + 1}, L = {L}, đáp án = {ans}")

trace([[1, 2], [1, 3], [3, 4], [3, 5]], [[1, 4], [3, 4], [2, 5], [4, 4], [5, 2]])

độ sâu (nhãn 1..n): {1: 0, 2: 1, 3: 1, 4: 2, 5: 2}
[1,4]: nâng 4 lên 2^1 -> 1 | lca = 1, L = 2, đáp án = 2
[3,4]: nâng 4 lên 2^0 -> 3 | lca = 3, L = 1, đáp án = 1
[2,5]: nâng 5 lên 2^0 -> 3 | lca = 1, L = 3, đáp án = 4
[4,4]: không cần nhảy | lca = 4, L = 0, đáp án = 0
[5,2]: nâng 5 lên 2^0 -> 3 | lca = 1, L = 3, đáp án = 4


Thêm một trace cho bảng DP chẵn lẻ để thấy quy luật `2^(L−1)` xuất hiện tự nhiên:

In [13]:
w = ways_table(5)
print(' L | chẵn | lẻ')
for L, (e, o) in enumerate(w):
    print(f"{L:>2} | {e:>4} | {o:>3}")

 L | chẵn | lẻ
 0 |    1 |   0
 1 |    1 |   1
 2 |    2 |   2
 3 |    4 |   4
 4 |    8 |   8
 5 |   16 |  16


## 9. Code nộp LeetCode

In [14]:
class Solution:
    def assignEdgeWeights(self, edges: List[List[int]], queries: List[List[int]]) -> List[int]:
        MOD = 10**9 + 7
        n = len(edges) + 1
        adj = [[] for _ in range(n)]
        for a, b in edges:
            adj[a - 1].append(b - 1)
            adj[b - 1].append(a - 1)
        par, dep = [0] * n, [0] * n
        seen = [False] * n
        seen[0] = True
        order = [0]
        for u in order:
            for v in adj[u]:
                if not seen[v]:
                    seen[v] = True
                    par[v], dep[v] = u, dep[u] + 1
                    order.append(v)
        LOG = max(1, (n - 1).bit_length())
        up = [par]
        for _ in range(1, LOG):
            pr = up[-1]
            up.append([pr[pr[x]] for x in range(n)])
        p2 = [1] * (n + 1)
        for i in range(1, n + 1):
            p2[i] = p2[i - 1] * 2 % MOD

        res = []
        for u, v in queries:
            a, b = u - 1, v - 1
            if dep[a] < dep[b]:
                a, b = b, a
            diff, j = dep[a] - dep[b], 0
            while diff:
                if diff & 1:
                    a = up[j][a]
                diff >>= 1
                j += 1
            if a != b:
                for j in range(LOG - 1, -1, -1):
                    if up[j][a] != up[j][b]:
                        a, b = up[j][a], up[j][b]
                a = par[a]
            L = dep[u - 1] + dep[v - 1] - 2 * dep[a]
            res.append(p2[L - 1] if L else 0)
        return res

for e, q, exp in examples:
    got = Solution().assignEdgeWeights(e, q)
    print(got, 'OK' if got == exp else 'SAI')

[0, 1] OK
[2, 1, 4] OK


## 10. Vì sao thuật toán đúng

1. **Rút gọn về `L`:** chỉ các cạnh trên đường được xét, mỗi cạnh độc lập chọn 1 hoặc 2 ⇒ đáp án chỉ phụ thuộc số cạnh `L`.
2. **Đếm:** tổng lẻ ⇔ số cạnh nhận 1 là lẻ. Với `L ≥ 1`, ánh xạ "lật trọng số của cạnh đầu tiên (1 ↔ 2)" là song ánh giữa cách gán tổng lẻ và tổng chẵn ⇒ mỗi loại đúng `2^L / 2 = 2^(L−1)`. Với `L = 0` tổng là 0 (chẵn) ⇒ 0 cách.
3. **`L` đúng:** đường đi duy nhất giữa hai đỉnh của cây đi qua LCA; số cạnh = (từ `u` lên LCA) + (từ `v` lên LCA) = `dep[u] + dep[v] − 2·dep[lca]`.
4. **LCA đúng:** binary lifting chuẩn; Tarjan offline đúng vì khi `u` hoàn tất và `v` đã hoàn tất trước, đại diện tập của `v` trỏ tới tổ tiên chung sâu nhất đang "mở" — chính là LCA.

## 11. Phân tích Time / Space Complexity

Gọi `n` số đỉnh, `q` số truy vấn.

| Cách | Time | Space |
|---|---|---|
| 1. BFS + liệt kê `2^L` | `O(q (n + 2^L L))` | `O(n)` |
| 2. DP + leo cha | `O(n + q h)` | `O(n)` |
| 3. Binary lifting + công thức | `O((n + q) log n)` | `O(n log n)` |
| 4. Tarjan offline | `O((n + q) α(n))` | `O(n + q)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [15]:
impls = {
    'climb': assign_climb,
    'lifting': assign_lifting,
    'tarjan': assign_tarjan,
    'submit': lambda e, q: Solution().assignEdgeWeights(e, q),
}
edge = [
    ([[1, 2]], [[2, 2], [2, 1]], [0, 1]),
    ([[1, 2], [2, 3], [3, 4]], [[1, 4], [4, 1], [2, 3]], [4, 4, 1]),
    ([[2, 1], [3, 1], [4, 1]], [[2, 3], [4, 4]], [2, 0]),    # hình sao, cạnh viết ngược
]
for e, q, exp in examples + edge:
    for name, f in impls.items():
        got = f(e, q)
        assert got == exp, (name, e, q, got, exp)
    assert assign_brute(e, q) == exp
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn (cây nhỏ, đường ngắn)

In [16]:
import random
random.seed(3559)

def rand_tree(n):
    perm = list(range(1, n + 1))
    random.shuffle(perm)
    es = []
    for i in range(1, n):
        a, b = perm[random.randrange(i)], perm[i]
        es.append([a, b] if random.random() < 0.5 else [b, a])
    return es

cnt = 0
for _ in range(400):
    n = random.randint(2, 12)
    e = rand_tree(n)
    q = [[random.randint(1, n), random.randint(1, n)] for _ in range(8)]
    exp = assign_brute(e, q)
    for name, f in impls.items():
        assert f(e, q) == exp, (name, e, q)
    cnt += 1
print('Khớp vét cạn trên', cnt, 'cây ngẫu nhiên.')

Khớp vét cạn trên 400 cây ngẫu nhiên.


### 12.3. Ca lớn (`n = q = 10^5`)

In [17]:
import time
random.seed(5)
n = 10**5
qs = [[random.randint(1, n), random.randint(1, n)] for _ in range(n)]
for name, e in [('ngẫu nhiên', rand_tree(n)), ('đường thẳng', [[i, i + 1] for i in range(1, n)])]:
    t = time.time(); a = Solution().assignEdgeWeights(e, qs); t1 = time.time() - t
    t = time.time(); b = assign_tarjan(e, qs); t2 = time.time() - t
    assert a == b
    print(f"{name:>11}: binary lifting {t1:.2f}s | Tarjan {t2:.2f}s | 3 đáp án đầu {a[:3]}")

 ngẫu nhiên: binary lifting 0.66s | Tarjan 0.83s | 3 đáp án đầu [33554432, 262144, 2097152]


đường thẳng: binary lifting 0.56s | Tarjan 0.60s | 3 đáp án đầu [312149016, 69963185, 848205008]


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Trả `2^(L−1)` cả khi `L = 0`

In [18]:
def wrong_L0(L):
    return pow(2, L - 1, MOD) if L >= 0 else 0      # SAI: L = 0 -> pow(2, -1, MOD) = nghịch đảo của 2

print('sai :', wrong_L0(0))
print('đúng:', 0)

sai : 500000004
đúng: 0


**Vì sao sai:** đường từ một đỉnh tới chính nó không có cạnh, tổng 0 là chẵn ⇒ 0 cách. Trong Python, `pow(2, -1, MOD)` còn trả về **nghịch đảo modulo** (500000004) — một số "trông hợp lệ" nhưng sai hoàn toàn.

### Lỗi 2 — Tính `L` bằng `|dep[u] − dep[v]|`

In [19]:
def wrong_depth_diff(edges, queries):
    n = len(edges) + 1
    par, dep, _, _ = bfs(n, edges)
    return [pow(2, abs(dep[u - 1] - dep[v - 1]) - 1, MOD) if dep[u - 1] != dep[v - 1] else 0
            for u, v in queries]

e, q, exp = examples[1]
print('sai :', wrong_depth_diff(e, q))     # [2,5]: |1-2| = 1 -> 1
print('đúng:', Solution().assignEdgeWeights(e, q))

sai : [2, 1, 1]
đúng: [2, 1, 4]


**Vì sao sai:** chỉ đúng khi một đỉnh là tổ tiên của đỉnh kia. Đường `2 → 1 → 3 → 5` đi qua LCA ở trên.

### Lỗi 3 — Tính `2^L` (đếm mọi cách gán) hoặc `2^L − 1`

In [20]:
for L in range(1, 5):
    odd = sum(1 for ws in product([1, 2], repeat=L) if sum(ws) % 2)
    print(f"L={L}: thật = {odd}, 2^L = {2 ** L}, 2^L - 1 = {2 ** L - 1}, 2^(L-1) = {2 ** (L - 1)}")

L=1: thật = 1, 2^L = 2, 2^L - 1 = 1, 2^(L-1) = 1
L=2: thật = 2, 2^L = 4, 2^L - 1 = 3, 2^(L-1) = 2
L=3: thật = 4, 2^L = 8, 2^L - 1 = 7, 2^(L-1) = 4
L=4: thật = 8, 2^L = 16, 2^L - 1 = 15, 2^(L-1) = 8


**Vì sao sai:** chỉ một nửa số cách gán cho tổng lẻ.

### Lỗi 4 — Đệ quy DFS để tính độ sâu trên cây đường thẳng

In [21]:
def depth_recursive(adj, u, p, d, dep):
    dep[u] = d
    for v in adj[u]:
        if v != p:
            depth_recursive(adj, v, u, d + 1, dep)

n = 3000
_, _, _, adj_chain = bfs(n, [[i, i + 1] for i in range(1, n)])
try:
    depth_recursive(adj_chain, 0, -1, 0, [0] * n)
    print('chạy được')
except RecursionError:
    print('RecursionError với cây đường thẳng 3000 đỉnh -> dùng BFS/stack như lời giải chính')

RecursionError với cây đường thẳng 3000 đỉnh -> dùng BFS/stack như lời giải chính


**Vì sao sai:** giới hạn đệ quy mặc định ~1000; `n` tới `10^5`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đếm cách gán {lẻ, chẵn} để tổng lẻ/chẵn" ⇒ đáp án thường là **một nửa** tổng số cách: `2^(L−1)`.
- "Truy vấn trên đường đi giữa hai đỉnh" ⇒ **LCA** + giá trị tiền tố (độ sâu, tổng trọng số).
- Nhiều truy vấn lũy thừa ⇒ dựng bảng `p2` một lần.
- Đọc được hết truy vấn trước ⇒ **Tarjan offline** là lựa chọn gọn, không cần `n log n` bộ nhớ.
- Cẩn thận `pow(x, -1, MOD)` trong Python ≥ 3.8 không báo lỗi.

## 15. Biến thể của bài

1. Bản I (LC 3558): chỉ hỏi đường từ gốc tới đỉnh sâu nhất ⇒ `L = max depth`.
2. Trọng số được chọn trong `{1, 2, 3}` ⇒ số cách tổng lẻ: lẻ có 2 lựa chọn (1, 3), chẵn có 1 ⇒ `(3^L − (−1)^L)/2`... (tính bằng DP chẵn lẻ).
3. Hỏi tổng chia hết cho `k` ⇒ DP theo phần dư `k` trạng thái, hoặc lũy thừa ma trận.
4. Cạnh có trọng số cố định một phần ⇒ đếm số cạnh tự do và chẵn lẻ của phần cố định.

In [22]:
# Biến thể 2: trọng số {1,2,3}, đếm tổng lẻ bằng DP và đối chiếu liệt kê
def ways_123(L):
    e, o = 1, 0
    for _ in range(L):
        e, o = e * 1 + o * 2, o * 1 + e * 2      # thêm 2 (giữ) ; thêm 1 hoặc 3 (đảo)
    return o

for L in range(0, 6):
    brute = sum(1 for ws in product([1, 2, 3], repeat=L) if sum(ws) % 2)
    assert ways_123(L) == brute == (3 ** L - (-1) ** L) // 2
print([ways_123(L) for L in range(6)])

[0, 2, 4, 14, 40, 122]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1 | BFS + liệt kê | mũ theo `L` | `O(n)` | Oracle |
| 2 | DP chẵn lẻ + leo cha | `O(n + qh)` | `O(n)` | Cây nông |
| 3 | Binary lifting + `2^(L−1)` | `O((n+q) log n)` | `O(n log n)` | **Nộp bài** (online) |
| 4 | Tarjan offline | `O((n+q) α)` | `O(n+q)` | Khi đọc trước được truy vấn |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** cây đường thẳng 1-2-3-4-5, truy vấn `[[1,5],[2,2],[5,4]]`.

**Bài 2 (tính tay):** `L = 20` ⇒ đáp án?

**Bài 3 (điền TODO):** viết `num_edges(u, v)` dùng LCA.

**Bài 4 (tự viết lại):** biến thể 3 với `k = 3`: đếm số cách gán `{1, 2}` để tổng **chia hết cho 3**.

**Bài 5 (mini challenge):** viết LCA bằng **Euler tour + sparse table** (`O(1)` mỗi truy vấn) và đối chiếu với binary lifting.

In [23]:
def num_edges_todo(up, dep, u, v):
    # TODO
    raise NotImplementedError

def ways_mod3_todo(L):
    # TODO: DP theo phần dư 0/1/2
    raise NotImplementedError

def lca_euler_todo(n, edges, queries):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `L = 4, 0, 1`. (2) `2^3, 0, 2^0`. (3) `[8, 0, 1]`.

**Bài 2.** (1) `2^19`. (2) `= 524288`. (3) Nhỏ hơn MOD nên giữ nguyên.

**Bài 3.** (1) `l = lca(u, v)`. (2) `dep[u] + dep[v] − 2·dep[l]`. (3) So với BFS.

**Bài 4.** (1) Trạng thái `r ∈ {0,1,2}`. (2) Thêm cạnh 1: `r → r+1`, cạnh 2: `r → r+2`. (3) Đáp án `cnt[0]`.

**Bài 5.** (1) Duyệt Euler ghi lại đỉnh mỗi lần đi qua (độ dài `2n − 1`). (2) LCA = đỉnh có độ sâu nhỏ nhất giữa `first[u]` và `first[v]`. (3) Sparse table theo cặp `(dep, đỉnh)`.

## 19. Đáp án tham khảo cho bài tập

In [24]:
# Bài 1, 2, 3
a1 = Solution().assignEdgeWeights([[1, 2], [2, 3], [3, 4], [4, 5]], [[1, 5], [2, 2], [5, 4]])
assert a1 == [8, 0, 1]
a2 = pow(2, 19, MOD)
assert a2 == 524288

def num_edges(up, dep, u, v):
    l = lca(up, dep, u, v)
    return dep[u] + dep[v] - 2 * dep[l]

random.seed(3)
for _ in range(100):
    n = random.randint(2, 15)
    e = rand_tree(n)
    par, dep, _, adj = bfs(n, e)
    up = build_up(par)
    for _ in range(10):
        u, v = random.randrange(n), random.randrange(n)
        assert num_edges(up, dep, u, v) == path_len_bfs(adj, u, v)
print('Bài 1:', a1, '| Bài 2:', a2, '| Bài 3: num_edges đúng')

Bài 1: [8, 0, 1] | Bài 2: 524288 | Bài 3: num_edges đúng


In [25]:
# Bài 4
def ways_mod3(L):
    cnt = [1, 0, 0]
    for _ in range(L):
        cnt = [(cnt[(r - 1) % 3] + cnt[(r - 2) % 3]) % MOD for r in range(3)]
    return cnt[0]

for L in range(0, 9):
    brute = sum(1 for ws in product([1, 2], repeat=L) if sum(ws) % 3 == 0)
    assert ways_mod3(L) == brute
print('Bài 4:', [ways_mod3(L) for L in range(9)])

Bài 4: [1, 0, 2, 2, 6, 10, 22, 42, 86]


In [26]:
# Bài 5: Euler tour + sparse table
def lca_euler(n, edges, queries):
    par, dep, order, adj = bfs(n, edges)
    children = [[] for _ in range(n)]
    for v in order[1:]:
        children[par[v]].append(v)
    euler, first = [], [0] * n
    stack = [(0, 0)]
    while stack:
        u, i = stack.pop()
        if i == 0:
            first[u] = len(euler)
        euler.append(u)
        if i < len(children[u]):
            stack.append((u, i + 1))
            stack.append((children[u][i], 0))
    m = len(euler)
    sp = [[(dep[x], x) for x in euler]]
    k = 1
    while (1 << k) <= m:
        prev = sp[-1]
        h = 1 << (k - 1)
        sp.append([min(prev[i], prev[i + h]) for i in range(m - (1 << k) + 1)])
        k += 1
    out = []
    for u, v in queries:
        l, r = sorted((first[u], first[v]))
        j = (r - l + 1).bit_length() - 1
        out.append(min(sp[j][l], sp[j][r - (1 << j) + 1])[1])
    return out

random.seed(4)
for _ in range(100):
    n = random.randint(2, 30)
    e = rand_tree(n)
    par, dep, _, _ = bfs(n, e)
    up = build_up(par)
    qs = [(random.randrange(n), random.randrange(n)) for _ in range(20)]
    assert lca_euler(n, e, qs) == [lca(up, dep, u, v) for u, v in qs]
print('Bài 5: LCA Euler tour khớp binary lifting')

Bài 5: LCA Euler tour khớp binary lifting


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
BFS từ gốc: par, dep;  up[j][x] = up[j-1][up[j-1][x]]
p2[i] = 2^i mod MOD
mỗi truy vấn: l = lca(u, v); L = dep[u] + dep[v] - 2*dep[l]
đáp án = p2[L-1] nếu L > 0 else 0
```

**Checklist:**

- [ ] Nhãn 1-based → 0-based?
- [ ] `L = 0` trả 0 (không dùng `pow(2, -1, MOD)`)?
- [ ] `L` dùng LCA, không phải hiệu độ sâu?
- [ ] Đếm **một nửa** số cách: `2^(L−1)`?
- [ ] Không đệ quy (cây đường thẳng `10^5`)?